# 5. Contact pieces (Section 5.3, Theorem 3) and complete light curves

**5a. Theorem 3, piece by piece.** pizzetti's proven mode splits the limb region into pieces, each
with a certified bound. Each piece is evaluated **on its own** (no neighbouring piece, no interior
series) at points spread over its range, ends included, and its error against the reference is
compared with its certified bound.

**5b. End to end.** `occult()` in the default (`fast`) and `proven` modes against the reference,
across the whole transit: every certified sample must be within `series_tol`.

In [1]:
import time
import numpy as np
import mpmath as mp
import refcheck as rc

FULL = False          # True: the larger grids (minutes to hours)
t_start = time.time()

from pizzetti import occult, series_cut, _proven
from pizzetti.occult import limb_tables
from pizzetti.laws import power_terms

LAWS = [("squareroot", [0.3, 0.4]), ("nonlinear", [0.6, -0.2, 0.3, -0.1]), ("power2", [0.6, 0.6]),
        ("power2", [0.7, 1.0]), ("power2", [0.5, 1.7])]
if not FULL:
    LAWS = [LAWS[1], LAWS[3]]
PS = [0.0092, 0.1, 0.2] if FULL else [0.0092, 0.2]
TAUS = [1e-9, 1e-12]


def blocked(z, p, al, c):
    return sum(ci * rc.P_fast(z, p, ai / 2) for ai, ci in zip(al, c))

## 5a. Theorem 3, piece by piece

In [2]:
npieces = 0
ratios = []
uncert = []
for law, u in LAWS:
    al, c = power_terms(law, u)
    g = al / 2
    T, L = limb_tables(g)
    total = 2 * np.pi * np.sum(c / (al + 2))
    for p in PS:
        for tau in TAUS:
            zc = series_cut(p, law, u, tau)
            zb, kinds, s0s, store, degs, ok, bnds = _proven.build(p, g, c, T, L, zc, total, tau)
            for pc in range(len(kinds)):
                if not np.isfinite(bnds[pc]):
                    uncert.append((law, p, tau))
                    continue
                a, b = zb[pc]
                if pc == 0:
                    b = 1 + p - 1e-12
                zz = np.concatenate([np.linspace(a, b, 7 if FULL else 4), [a + (b - a) * 1e-9, b - (b - a) * 1e-9]])
                f = np.empty(zz.size)
                sl = slice(pc, pc + 1)
                _proven.evaluate(zz, p, g, zb[sl], kinds[sl], s0s[sl], store[sl], degs[sl], f)
                err = max(abs(fi - blocked(zi, p, al, c)) for fi, zi in zip(f, zz)) / total
                ratios.append(err / bnds[pc])
                npieces += 1
print(f"certified pieces checked: {npieces}; error / bound: max {max(ratios):.3f}, median {np.median(ratios):.3f}")
print(f"pieces without a certificate (computed by quadrature instead): {len(uncert)} {sorted(set(uncert))}")
assert max(ratios) <= 1

certified pieces checked: 47; error / bound: max 0.956, median 0.673
pieces without a certificate (computed by quadrature instead): 4 [('nonlinear', 0.2, 1e-12), ('power2', 0.2, 1e-12)]


## 5b. Complete light curves

In [3]:
worst = {"fast": 0.0, "proven": 0.0}
for law, u in LAWS:
    al, c = power_terms(law, u)
    total = 2 * np.pi * np.sum(c / (al + 2))
    for p in PS:
        for tau in TAUS:
            zc = series_cut(p, law, u, tau)
            d = np.array([1e-10, 1e-5])
            zs = np.unique(np.concatenate([np.linspace(0, 1 + p, 30 if FULL else 12)[1:-1], [zc - 1e-9, zc + 1e-9],
                                           1 - p - d * p, 1 - p + d * p, 1 + p - d * p]))
            ref = np.array([1 - blocked(z, p, al, c) / total for z in zs])
            for mode in ["fast", "proven"]:
                f, cert = occult(zs, p, u, law, series_tol=tau, limb=mode, return_certified=True)
                e = np.abs(f - ref)
                worst[mode] = max(worst[mode], np.max(e) / tau)
                assert np.all(e[cert] <= tau * (1 + 1e-6) + 1e-15), (law, p, tau, mode)
for m, v in worst.items():
    print(f"{m:>6}: largest error / series_tol = {v:.3f}")

  fast: largest error / series_tol = 0.926
proven: largest error / series_tol = 0.926


In [4]:
print(f"notebook run time: {time.time() - t_start:.0f} s")

notebook run time: 8 s
